In [61]:
# functions
import numpy as np
from scipy.stats import entropy
def jsd_multiple(distributions, weights=None, base=2):
    """
    Generalized Jensen-Shannon divergence across N >= 2 distributions.

    distributions: array of shape (n_distributions, n_categories),
                    each row summing to 1
    weights: optional weights for each distribution (e.g. relative sample
              sizes); defaults to equal weighting
    base: log base for entropy (2 = bits, np.e = nats)

    JSD = H(weighted mean distribution) - weighted_mean(H(each distribution))
    """
    distributions = np.asarray(distributions, dtype=float)
    n = distributions.shape[0]

    if weights is None:
        weights = np.full(n, 1.0 / n)
    else:
        weights = np.asarray(weights, dtype=float)
        weights = weights / weights.sum()  # normalize just in case

    mean_dist = np.average(distributions, axis=0, weights=weights)

    entropy_of_mean = entropy(mean_dist, base=base)
    mean_of_entropies = np.average(
        [entropy(d, base=base) for d in distributions], weights=weights
    )

    return entropy_of_mean - mean_of_entropies
def format_data(file_name,language =None, modifiers=None):
    if language == None:
        # return message to add language
        print("Please specify a language: 'EN' or 'JP'")
        return
    df = read_csv(file_name)
    df = df[df.columns[df.columns.isin(["doc_id", "predicate", "attitude", "relationship", "modifier_response_list"])]]
    if language == "EN":
        df = df.assign(
            modifier=df["modifier_response_list"].str.split(";")
        ).explode("modifier").reset_index(drop=True)
    if language == "JP":
        df = df.assign(
            modifier=df["modifier_response_list"].str.split(r"[;\s\u3000]+")
        ).explode("modifier").reset_index(drop=True)
    # make each modifier in the modifier response list a separate row, so if there are three modifiers in the list, there will be three rows with the same doc_id, predicate, attitude, relationship, and is_negated value
    df["modifier"] = df["modifier"].str.strip()

    # take out these columns: doc_id,predicate,attitude,relationship,modifier_response_list where modifier list is a string like none; too; very; pretty; really; quite 
    df["is_negated"] = df.apply(lambda x: (x["attitude"], x["predicate"]) in NEGATED_SET, axis=1)
    df["modifier"] = df["modifier"].replace(MODIFIER_VARIANTS_FIX)
    df.drop(columns=["modifier_response_list"], inplace=True)
    if modifiers is not None:
    # filter rows where modifier is not in modifiers
        # print the percent of rows dropped and the first 10 most common modifiers that were dropped
        dropped_rows = df[~df["modifier"].isin(modifiers)]
        percent_dropped = len(dropped_rows) / len(df) * 100
        print(f"Dropped {len(dropped_rows)} rows ({percent_dropped:.2f}%) because their modifiers were not in the specified list.")
        if len(dropped_rows) > 0:
            print("Most common dropped modifiers:")
            # if super is the modifier, show sentences
            if "super" in dropped_rows["modifier"].values:
                print("Sentences with 'super':")
                print(dropped_rows[dropped_rows["modifier"] == "super"][["doc_id", "predicate", "attitude", "relationship"]])
            print(dropped_rows["modifier"].value_counts().head(10))
        return df[df["modifier"].isin(modifiers)]
    return df
def filter_axis(axis,MODIFIERS,df):
    modifier_distribution = df.groupby([axis, "modifier"]).size().reset_index(name="count")
    modifier_distribution = modifier_distribution.pivot(index=axis, columns="modifier", values="count")
    modifier_distribution = modifier_distribution.reindex(columns=MODIFIERS, fill_value=0)  # ensure ALL modifiers present, even unobserved ones
    modifier_distribution.fillna(0, inplace=True)  # fill NaN with 0 for unobserved modifiers
    return modifier_distribution.div(modifier_distribution.sum(axis=1), axis=0)

In [68]:
JP_MODIFIERS = ["あまり", "いまいち", "かなり", "すごく", "そこまで", "それほど", "そんなに", "たいして", "だいぶ", "ちっとも", "ちょっと", "とても", "なかなか", "なし", "ひどく", "まあまあ", "マジで", "めっちゃ", "やや", "結構", "若干", "少し", "少しも", "全然", "相当", "超", "非常に", "微妙に", "普通に", "本当に", "全く","別に","あんまり"]
EN_MODIFIERS = ["very", "really", "that", "quite", "none", "too", "slightly", "as","super", "somewhat", "pretty", "a little bit", "at all", "kind of", "completely", "kinda", "a bit", "semi", "a tad", "incredibly", "sorta", "totally", "a little", "amazingly", "extremely", "moderately","mildly",  "clearly", "damn", "majorly",  "absolutely", "exceptionally","so"]
NEGATED_SET = {
    ("non-committal", "面白い"),
    ("non-committal", "美味しい"),
    ("non-committal", "綺麗"),
    ("warning",       "面白い"),
    ("warning",       "美味しい"),
    ("warning",       "綺麗"),
    ("annoyed",       "面白い"),
    ("annoyed",       "美味しい"),
    ("annoyed",       "綺麗"),
    ("encouraging",   "遅れている"),
    ("encouraging",   "寒い"),
    ("encouraging",   "汚い"),
    ("acknowledge",   "遅れている"),
    ("acknowledge",   "寒い"),
    ("acknowledge",   "汚い"),
}
MODIFIER_VARIANTS_FIX = {
    # 表記ゆれ (kanji / kana / katakana)
    "凄く": "すごく",
    "ぜんぜん": "全然",
    "まったく": "全く",
    "大分": "だいぶ",
    "そうとう": "相当",
    "けっこう": "結構",
    "ほんとに": "本当に",
    "すこし": "少し",
    "イマイチ": "いまいち",
    "widly": "wildly",
    "sasdly": "sadly",
    "unfortunely": "unfortunately",
    "thaat": "that",
    "little\\": "little"
}

## print out JSD results

In [69]:
# read in csv files and create a file that 
from pandas import read_csv
en_df = format_data("/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPolitenessOfModifiers/data_analysis/EN_trials.csv","EN" ,EN_MODIFIERS)
jp_df = format_data("/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPolitenessOfModifiers/data_analysis/JP_trials.csv", "JP", JP_MODIFIERS)

# change wording to unify different variants: 
en_modifier_dist = filter_axis("relationship", EN_MODIFIERS,en_df)
jp_modifier_dist = filter_axis("relationship", JP_MODIFIERS,jp_df)
print("JSD across relationships in English:", jsd_multiple(en_modifier_dist.values))
print("JSD across relationships in Japanese:", jsd_multiple(jp_modifier_dist.values))
en_modifier_dist = filter_axis("attitude", EN_MODIFIERS,en_df)
jp_modifier_dist = filter_axis("attitude", JP_MODIFIERS,jp_df)
print("JSD across attitudes in English:", jsd_multiple(en_modifier_dist.values))
print("JSD across attitudes in Japanese:", jsd_multiple(jp_modifier_dist.values))
# same thing but when en_df and jp_df are filtered to only include rows where is_negated is False
en_df_pos = en_df[en_df["is_negated"] == False]
jp_df_pos = jp_df[jp_df["is_negated"] == False]
en_modifier_dist = filter_axis("relationship", EN_MODIFIERS,en_df_pos)
jp_modifier_dist = filter_axis("relationship", JP_MODIFIERS,jp_df_pos)
print("JSD across relationships in English (negated filtered):", jsd_multiple(en_modifier_dist.values))
print("JSD across relationships in Japanese (negated filtered):", jsd_multiple(jp_modifier_dist.values))
en_modifier_dist = filter_axis("attitude", EN_MODIFIERS,en_df_pos)
jp_modifier_dist = filter_axis("attitude", JP_MODIFIERS,jp_df_pos)
print("JSD across attitudes in English (negated filtered):", jsd_multiple(en_modifier_dist.values))
print("JSD across attitudes in Japanese (negated filtered):", jsd_multiple(jp_modifier_dist.values))

# same thing but when en_df and jp_df are filtered to only include rows where is_negated is True
en_df_neg = en_df[en_df["is_negated"] == True]
jp_df_neg = jp_df[jp_df["is_negated"] == True]
en_modifier_dist = filter_axis("relationship", EN_MODIFIERS,en_df_neg)
jp_modifier_dist = filter_axis("relationship", JP_MODIFIERS,jp_df_neg)
print("JSD across relationships in English (negated only):", jsd_multiple(en_modifier_dist.values))
print("JSD across relationships in Japanese (negated only):", jsd_multiple(jp_modifier_dist.values))
en_modifier_dist = filter_axis("attitude", EN_MODIFIERS,en_df_neg)
jp_modifier_dist = filter_axis("attitude", JP_MODIFIERS,jp_df_neg)
print("JSD across attitudes in English (negated only):", jsd_multiple(en_modifier_dist.values))
print("JSD across attitudes in Japanese (negated only):", jsd_multiple(jp_modifier_dist.values))

Dropped 147 rows (19.52%) because their modifiers were not in the specified list.
Most common dropped modifiers:
modifier
not              13
even              7
sadly             7
unfortunately     7
insanely          6
particularly      5
surprisingly      5
massively         5
wildly            4
actually          4
Name: count, dtype: int64
Dropped 398 rows (37.55%) because their modifiers were not in the specified list.
Most common dropped modifiers:
modifier
思ったより     7
最高に       5
ちゃんと      5
思った以上に    5
言うほど      5
そのため      5
予想以上に     5
もしかしたら    5
気にするほど    5
なので       4
Name: count, dtype: int64
JSD across relationships in English: 0.0972459771121934
JSD across relationships in Japanese: 0.09849172812647744
JSD across attitudes in English: 0.36573949778325066
JSD across attitudes in Japanese: 0.7090899622904274
JSD across relationships in English (negated filtered): 0.14986691947964115
JSD across relationships in Japanese (negated filtered): 0.10219044038515879
JSD across 

In [41]:
en_df = format_data("/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPolitenessOfModifiers/data_analysis/EN_trials.csv")
jp_df = format_data("/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPolitenessOfModifiers/data_analysis/JP_trials.csv")
# print all modifiers in each of the dataframes
print("English modifiers:", en_df["modifier"].unique())
print("Japanese modifiers:", jp_df["modifier"].unique())

English modifiers: ['then' 'very' 'quite' 'really' 'none' 'pretty' 'surprisingly' 'so' 'too'
 'even' 'that' 'particularly' 'slightly' 'extremely' 'super' 'incredibly'
 'as' 'still' 'therefore' 'especially' 'not' 'never' 'rarely' 'always'
 'feeling' 'getting' 'actually' 'disgustingly' 'totally' 'rather'
 'unbearably' 'unfathomably' 'obviously' 'thus' 'unfortunately' 'severely'
 'massively' 'completely' 'horrendously' 'clearly' 'terribly' 'at' 'all'
 'a' 'little' 'bit' 'tad' 'minorly' 'sadly' 'perfectly' 'thankfully'
 'amazingly' 'majorly' 'George' 'he' 'regrettably' 'fucking' 'insanely'
 'wildly' 'crazy' 'much' 'horribly' 'kinda' 'somewhat' 'hilariously'
 'indescribably' 'freezing' 'ice' 'deliciously' 'gross' 'overly' 'nasty'
 'damn' 'sorta' 'greatly' 'oddly' 'semi' 'ghastly' 'vastly' 'slowly'
 'moderately' 'exceptionally' 'shockingly' 'drastically' 'freakishly'
 'freaking' 'hella' 'awfully' 'smelly' 'stinky' 'piggy' 'bitterly'
 'seemingly' 'absolutely' 'mildly' 'kind' 'of' 'awful' 'won

## Compute the CI of JSD Japanese - JSD English and p-value that JSD Japanese > JSD English

In [71]:
"""
Bare-minimum bootstrap CI for JSD(attitudes), English vs Japanese.
Paste into the existing notebook; uses its jsd_multiple() and *_df / *_MODIFIERS.

ONE FIX FIRST, in format_data():
    df["modifier"] = df["modifier"].str.strip()
before the .isin(modifiers) filter. Splitting on ";" leaves a leading space on every
modifier after the first, so "; "-separated cells silently lose all but the first.

FLOOR CORRECTION, English vs Japanese: JP_MODIFIERS is a larger inventory than
EN_MODIFIERS, and a larger modifier inventory inflates JSD upward by itself (more
categories -> more finite-sample noise even under H0 of "no real attitude effect").
So raw JSD is not comparable across languages: JP will look bigger partly just
because it has more modifier categories, not because attitude matters more there.

Fix: estimate each language's own chance floor by permuting the attitude label
within participant (labels shuffled, so any true attitude effect is destroyed,
but inventory size / participant structure / sample size are preserved exactly).
Then correct each bootstrap draw by that language's floor and headroom
(ceiling = log2(#levels), same for both languages here since attitude has the
same number of levels in both) before differencing JP vs EN. This puts both
languages on a comparable 0-1-ish scale so the delta reflects a real difference
in attitude signal, not a difference in modifier-inventory size.
"""

import numpy as np


def boot_jsd(df, MODIFIERS, part_col, axis="attitude", n_boot=2000, n_perm=500, seed=0):
    """Resample PARTICIPANTS with replacement; each drawn participant contributes
    all their attitudes, preserving within-participant correlation.

    Also estimates this language's own chance floor (via label permutation within
    participant) so the returned bootstrap distribution can be floor-normalized
    before comparing across languages with different-sized modifier inventories."""
    df = df[df[axis].notna() & df[part_col].notna() & df["modifier"].notna()]
    mix = {m: i for i, m in enumerate(MODIFIERS)}
    participants = sorted(df[part_col].astype(str).unique())
    levels = sorted(df[axis].astype(str).unique())
    pix = {p: i for i, p in enumerate(participants)}
    aix = {a: i for i, a in enumerate(levels)}
    K = len(levels)

    counts = np.zeros((len(participants), len(levels), len(MODIFIERS)))   # dimensions is [participant, attitude, modifier]
    np.add.at(counts, (df[part_col].astype(str).map(pix).to_numpy(),
                       df[axis].astype(str).map(aix).to_numpy(),
                       df["modifier"].map(mix).to_numpy()), 1)

    def jsd_of(Mat):
        t = Mat.sum(axis=1); k = t > 0 # sums over all modifiers for each attitude
        return jsd_multiple(Mat[k] / t[k, None]) # Mat[k] / t[k, None] gives P(modifier | attitude) for each attitude

    rng = np.random.default_rng(seed)
    empirical = jsd_of(counts.sum(axis=0))

    # counts[rng.integers(0, len(participants), len(participants))] takes a bootstrap sample of participants, and .sum(axis=0) sums their counts across attitudes and modifiers. Then we compute the JSD of that summed distribution.
    bootstrapped_results = np.array([jsd_of(counts[rng.integers(0, len(participants), len(participants))].sum(axis=0))
                     for _ in range(n_boot)])

    # --- chance floor for this language/inventory: shuffle attitude labels
    #     within each participant (breaks any real attitude effect but keeps
    #     inventory size M and per-participant totals exactly as observed) ---
    perm_results = np.empty(n_perm)
    for t in range(n_perm):
        shuffled = np.empty_like(counts)
        for p in range(len(participants)):
            perm = rng.permutation(K)
            shuffled[p] = counts[p, perm]
        perm_results[t] = jsd_of(shuffled.sum(axis=0))
    floor = float(np.nanmean(perm_results))
    ceiling = float(np.log2(K))
    headroom = ceiling - floor

    return empirical, bootstrapped_results, floor, headroom


def delta_ci(en_df, jp_df, EN_MODIFIERS, JP_MODIFIERS, part_col,
             axis="attitude", n_boot=2000, n_perm=500, seed=0):
    e_obs, e_boot, e_floor, e_head = boot_jsd(en_df, EN_MODIFIERS, part_col, axis, n_boot, n_perm, seed)
    j_obs, j_boot, j_floor, j_head = boot_jsd(jp_df, JP_MODIFIERS, part_col, axis, n_boot, n_perm, seed)

    # raw (uncorrected) delta, kept for reference -- inflated by JP's larger inventory
    d_raw = j_boot - e_boot
    lo_raw, hi_raw = np.percentile(d_raw, [2.5, 97.5])

    # floor-normalized delta: each language's draws rescaled to (jsd - floor) / headroom
    # before differencing, so inventory-size bias cancels out
    e_norm = (e_boot - e_floor) / e_head
    j_norm = (j_boot - j_floor) / j_head
    d = j_norm - e_norm
    lo, hi = np.percentile(d, [2.5, 97.5])
    frac = (d <= 0).mean()
    p = max(min(1.0, 2 * min(frac, 1 - frac)), 1 / (n_boot + 1))

    print(f"  EN floor {e_floor:.4f}, headroom {e_head:.4f}  |  "
          f"JP floor {j_floor:.4f}, headroom {j_head:.4f}")
    print(f"Delta raw JSD (JP - EN) 95% CI     = [{lo_raw:+.4f}, {hi_raw:+.4f}]   <- inventory-size-biased, for reference only")
    print(f"Delta normalized (JP - EN) 95% CI  = [{lo:+.4f}, {hi:+.4f}]   "
          f"p {'< ' + format(1/(n_boot+1), '.2g') if frac in (0.0, 1.0) else '= ' + format(p, '.4f')}")
    return lo, hi

delta_ci(en_df, jp_df, EN_MODIFIERS, JP_MODIFIERS, part_col="doc_id", axis="attitude", n_boot=2000, seed=0)
delta_ci(en_df[en_df["is_negated"] == False], jp_df[jp_df["is_negated"] == False], EN_MODIFIERS, JP_MODIFIERS, part_col="doc_id", axis="attitude", n_boot=2000, seed=0)
delta_ci(en_df[en_df["is_negated"] == True], jp_df[jp_df["is_negated"] == True], EN_MODIFIERS, JP_MODIFIERS, part_col="doc_id", axis="attitude", n_boot=2000, seed=0)

  EN floor 0.2410, headroom 2.3440  |  JP floor 0.2852, headroom 2.2998
Delta raw JSD (JP - EN) 95% CI     = [+0.1167, +0.5961]   <- inventory-size-biased, for reference only
Delta normalized (JP - EN) 95% CI  = [+0.0345, +0.2404]   p = 0.0130
  EN floor 0.3161, headroom 2.2689  |  JP floor 0.2786, headroom 2.3064
Delta raw JSD (JP - EN) 95% CI     = [-0.0957, +0.4961]   <- inventory-size-biased, for reference only
Delta normalized (JP - EN) 95% CI  = [-0.0284, +0.2311]   p = 0.1080
  EN floor 0.2561, headroom 2.0658  |  JP floor 0.3085, headroom 2.0135
Delta raw JSD (JP - EN) 95% CI     = [+0.1247, +0.5370]   <- inventory-size-biased, for reference only
Delta normalized (JP - EN) 95% CI  = [+0.0396, +0.2410]   p = 0.0070


(np.float64(0.03963369070808495), np.float64(0.24096490933436032))

# check attitude and relationship changes language use

In [ ]:

"""
Are the P(modifier | level) distributions different across levels of one axis?
 
    H0: all levels of `axis` share the same modifier distribution
 
CROSSED DESIGN.  Each participant sees one stimulus for every
attitude x relationship combination, so every participant appears at every level
of both axes. That has two consequences, and getting either wrong costs a lot:
 
1. PERMUTE WITHIN (participant x other_axis).  Under H0 a participant's cells at
   different attitude levels are interchangeable *once relationship is held
   fixed*. Permuting globally across cells (or even freely within a participant)
   lets participant and relationship differences leak into the null and inflates
   it, making the test badly conservative -- median p ~0.8 instead of ~0.5 in
   simulation, i.e. real effects get missed.
 
2. BOOTSTRAP PARTICIPANTS WITH ONE SHARED DRAW.  A drawn participant carries all
   their levels at once, so their personal style cancels across levels the way it
   does in the real data. Resampling each level independently breaks that and
   gives intervals ~1.15x too wide.
 
Raw JSD is non-negative and biased upward, so it is not an effect size: identical
distributions still produce a positive value. The permutation null's mean is that
chance floor, measured directly, so the effect is (observed - floor).
 
    from axis_test import test_axis
    test_axis(en_df, EN_MODIFIERS, "English: attitude", part_col="participant_id",
              axis="attitude", other_axis="relationship")
    test_axis(en_df, EN_MODIFIERS, "English: relationship", part_col="participant_id",
              axis="relationship", other_axis="attitude")
"""
 
import numpy as np
 
 
def _build(df, MODIFIERS, part_col, axis, other_axis, verbose=True):
    """-> counts[participant, other_level, axis_level, modifier]."""
    need = [part_col, axis, other_axis, "modifier"]
    df = df[np.logical_and.reduce([df[c].notna() for c in need])].copy()
    for c in [part_col, axis, other_axis]:
        df[c] = df[c].astype(str)
 
    parts = sorted(df[part_col].unique())
    lev = sorted(df[axis].unique())
    oth = sorted(df[other_axis].unique())
    ix = [{v: i for i, v in enumerate(vs)} for vs in (parts, oth, lev)]
    mix = {m: i for i, m in enumerate(MODIFIERS)}
 
    C = np.zeros((len(parts), len(oth), len(lev), len(MODIFIERS)))
    np.add.at(C, (df[part_col].map(ix[0]).to_numpy(),
                  df[other_axis].map(ix[1]).to_numpy(),
                  df[axis].map(ix[2]).to_numpy(),
                  df["modifier"].map(mix).to_numpy()), 1)
    if verbose:
        filled = (C.sum(-1) > 0).mean()
        print(f"  {len(parts)} participants x {len(oth)} {other_axis} x "
              f"{len(lev)} {axis}, M={len(MODIFIERS)}")
        print(f"  cells with data: {filled:.1%}"
              + ("" if filled > 0.95 else "   <- unbalanced; permutation moves empty"
                                          " cells around, which weakens the test"))
    return C, lev, oth, parts
 
 
def _jsd(Mat):
    """Mat: (levels, M) counts -> generalized JSD in bits, equal weight per level."""
    t = Mat.sum(-1)
    k = t > 0
    if k.sum() < 2:
        return np.nan
    return jsd_multiple(Mat[k] / t[k, None])
 
 
def test_axis(df, MODIFIERS, label, part_col, axis="attitude",
              other_axis="relationship", n_perm=2000, n_boot=1000,
              seed=0, alpha=0.05, verbose=True):
    print(f"[{label}]")
    C, lev, oth, parts = _build(df, MODIFIERS, part_col, axis, other_axis, verbose)
    P, O, K, M = C.shape
    rng = np.random.default_rng(seed)
 
    obs = _jsd(C.sum(axis=(0, 1)))
 
    # --- permutation: reshuffle the axis levels inside each (participant, other) --
    null = np.empty(n_perm)
    for t in range(n_perm):
        pm = np.argsort(rng.random((P, O, K)), axis=2)          # a permutation per cell-block
        null[t] = _jsd(np.take_along_axis(C, pm[..., None], axis=2).sum(axis=(0, 1)))
    floor = float(np.nanmean(null))
    effect = obs - floor
    n_ge = int((null >= obs).sum())
    p = (n_ge + 1) / (n_perm + 1)
    p_floor = 1.0 / (n_perm + 1)
    sd = float(np.nanstd(null, ddof=1))
 
    # --- CI: one shared participant draw; width only, recentred on `effect` ------
    raw = np.empty(n_boot)
    for b in range(n_boot):
        raw[b] = _jsd(C[rng.integers(0, P, P)].sum(axis=(0, 1)))
    mid = float(np.nanmedian(raw))
    lo_off, hi_off = np.nanpercentile(raw, [100 * alpha / 2, 100 * (1 - alpha / 2)]) - mid
    lo, hi = effect + lo_off, effect + hi_off
 
    pstr = f"< {p_floor:.2g}" if n_ge == 0 else f"= {p:.4f}"
    print(f"  observed JSD        {obs:.4f} bits")
    print(f"  chance floor        {floor:.4f} bits   <- identical {axis} levels give this")
    print(f"  effect (obs-floor)  {effect:.4f} bits")
    print(f"  95% CI              [{lo:+.4f}, {hi:+.4f}]")
    print(f"  p                   {pstr}   ({n_ge}/{n_perm} permutations reached it)")
    print(f"  null: mean {floor:.4f}, sd {sd:.4f};  observed is {(obs-floor)/sd:.1f} SDs out")
    print(f"  -> {f'{axis} levels DO differ' if (n_ge == 0 or p < alpha) else 'no detectable difference'}"
          f"   (ceiling log2({K}) = {np.log2(K):.3f} bits)\n")
 
    return dict(label=label, obs=obs, floor=floor, effect=effect, ci=(lo, hi),
                p=p, p_censored=(n_ge == 0), null=null, boots=raw, levels=lev)

test_axis(jp_df, JP_MODIFIERS, "Japanese: attitude", part_col="doc_id",
          axis="attitude", other_axis="relationship")
test_axis(jp_df, JP_MODIFIERS, "Japanese: relationship", part_col="doc_id",
          axis="relationship", other_axis="attitude")

[Japanese: attitude]
  10 participants x 4 relationship x 6 attitude, M=31
  cells with data: 82.5%   <- unbalanced; permutation moves empty cells around, which weakens the test
  observed JSD        0.8528 bits
  chance floor        0.2372 bits   <- identical attitude levels give this
  effect (obs-floor)  0.6156 bits
  95% CI              [+0.4775, +0.8074]
  p                   < 0.0005   (0/2000 permutations reached it)
  null: mean 0.2372, sd 0.0280;  observed is 22.0 SDs out
  -> attitude levels DO differ   (ceiling log2(6) = 2.585 bits)

[Japanese: relationship]
  10 participants x 6 attitude x 4 relationship, M=31
  cells with data: 82.5%   <- unbalanced; permutation moves empty cells around, which weakens the test
  observed JSD        0.1279 bits
  chance floor        0.1141 bits   <- identical relationship levels give this
  effect (obs-floor)  0.0139 bits
  95% CI              [-0.0403, +0.1215]
  p                   = 0.2019   (403/2000 permutations reached it)
  null: mea

{'label': 'Japanese: relationship',
 'obs': np.float64(0.1279319746334675),
 'floor': 0.11405184926281144,
 'effect': np.float64(0.013880125370656068),
 'ci': (np.float64(-0.04026972743510908), np.float64(0.12150817763724985)),
 'p': 0.20189905047476261,
 'p_censored': False,
 'null': array([0.11561451, 0.11758202, 0.08058018, ..., 0.09688762, 0.11797827,
        0.11694554]),
 'boots': array([0.17036242, 0.15304689, 0.16552441, 0.16099191, 0.23263422,
        0.19843806, 0.21080026, 0.20713988, 0.15547912, 0.16435491,
        0.17707134, 0.13314288, 0.20400821, 0.1783943 , 0.25581384,
        0.17817116, 0.26802209, 0.22426942, 0.20041982, 0.19412507,
        0.20191515, 0.26345923, 0.169877  , 0.17141499, 0.27802003,
        0.22821749, 0.20187042, 0.16949498, 0.17684958, 0.15276213,
        0.18050027, 0.17917618, 0.16703215, 0.13588563, 0.21482285,
        0.16111762, 0.23197508, 0.16493017, 0.16354461, 0.27892851,
        0.2300605 , 0.15448628, 0.21538102, 0.19294426, 0.15959279,

## see if attitude and relationship are confounding in changing modifier use

In [ ]:
"""
Is the attitude effect confounded with the relationship effect?

X = modifier, A = attitude, R = relationship. All quantities in bits.

    I(A;X)     = H(X) - H(X|A)          marginal attitude effect (pool over R)
    I(R;X)     = H(X) - H(X|R)          marginal relationship effect (pool over A)
    I(A,R;X)   = H(X) - H(X|A,R)        JSD across all A x R cells
    I(A;X|R)   = H(X|R) - H(X|A,R)      attitude effect HOLDING relationship fixed
    I(R;X|A)   = H(X|A) - H(X|A,R)      relationship effect holding attitude fixed

Chain rule (verified numerically below):
    I(A,R;X) = I(A;X) + I(R;X|A) = I(R;X) + I(A;X|R)

TWO DIAGNOSTICS

  I(A;R)  -- dependence between the two factors in the REALIZED data, computed on
             cell counts alone. Zero for a balanced crossed design. Non-zero means
             the factors are entangled and confounding is possible at all. This is
             the number to look at first; NEGATED_SET filtering is what makes it
             non-zero, since it deletes specific (attitude, predicate) pairs.

  II = I(A;X) + I(R;X) - I(A,R;X) = I(A;X) - I(A;X|R)
             > 0  redundancy: A and R carry overlapping information about X
             < 0  synergy: the pair says more than the parts
             = 0  additive

WHY THE SIGN IS INFORMATIVE ONLY WHEN I(A;R) > 0
  If A and R are independent by design, then
      I(A;X|R) - I(A;X) = I(A;R|X) - I(A;R) = I(A;R|X) >= 0
  so conditioning on R can only RAISE the attitude estimate. That is not bias being
  removed, it is relationship-induced noise being removed -- a power gain. Reading it
  as "confounding" would be wrong. Only once I(A;R) > 0 can conditioning move the
  estimate down, which is the signature of genuine confounding.

EVERY QUANTITY IS BIASED UPWARD at finite n, so each is reported against its own
permutation floor. Compare floor-corrected values, never raw ones.

    from decompose import decompose
    decompose(counts)        # counts[attitude, relationship, modifier]
"""

import numpy as np

LN2 = np.log(2.0)


def _H(p):
    p = np.asarray(p, float)
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / LN2)


def _cond_H(counts, axes):
    """H(X | the axes NOT summed out), weighted by empirical cell sizes."""
    c = counts.sum(axis=axes) if axes else counts       # -> (..., M)
    c = c.reshape(-1, counts.shape[-1])
    n = c.sum()
    tot = c.sum(axis=1)
    return float(sum((t / n) * _H(row / t) for row, t in zip(c, tot) if t > 0))


def quantities(counts):
    """counts[a, r, m] -> dict of entropies and mutual informations, in bits."""
    n = counts.sum()
    H_X = _H(counts.sum(axis=(0, 1)) / n)
    H_X_A = _cond_H(counts, (1,))          # condition on attitude
    H_X_R = _cond_H(counts, (0,))          # condition on relationship
    H_X_AR = _cond_H(counts, ())           # condition on both

    I_AX = H_X - H_X_A
    I_RX = H_X - H_X_R
    I_ARX = H_X - H_X_AR
    I_AX_R = H_X_R - H_X_AR
    I_RX_A = H_X_A - H_X_AR

    # factor dependence, from cell counts only
    w = counts.sum(axis=2) / n                          # (A, R) joint
    wa, wr = w.sum(axis=1), w.sum(axis=0)
    I_AR = _H(wa) + _H(wr) - _H(w.ravel())

    return dict(H_X=H_X, H_X_A=H_X_A, H_X_R=H_X_R, H_X_AR=H_X_AR,
                I_AX=I_AX, I_RX=I_RX, I_ARX=I_ARX,
                I_AX_R=I_AX_R, I_RX_A=I_RX_A,
                II=I_AX + I_RX - I_ARX, I_AR=I_AR)


def _shuffle_table(tab, rng):
    """tab: (G, M) counts -> a table with the SAME row and column margins but the
    group label randomly reassigned at the TOKEN level.

    Shuffling aggregated rows would be a no-op: mutual information depends only on
    the multiset of rows, so permuting which row carries which label leaves it
    unchanged. The upward bias comes from finite-sample noise in individual
    observations, so the tokens themselves have to be reassigned.
    """
    rows = tab.sum(axis=1).astype(np.int64)
    tokens = np.repeat(np.arange(tab.shape[1]), tab.sum(axis=0).astype(np.int64))
    rng.shuffle(tokens)
    out = np.zeros_like(tab)
    start = 0
    for g, k in enumerate(rows):
        np.add.at(out[g], tokens[start:start + k], 1)
        start += k
    return out


def _floors(counts, n_perm, seed):
    """Chance floors: what each quantity reads when the label carries no
    information, with all margins held at their observed values.
      I(A;X)   : reassign attitude among tokens, pooled over R
      I(A;X|R) : reassign attitude among tokens WITHIN each relationship
      I(R;X)   : reassign relationship among tokens, pooled over A
      I(R;X|A) : reassign relationship among tokens WITHIN each attitude
      I(A,R;X) : reassign the joint cell label among all tokens
    """
    rng = np.random.default_rng(seed)
    A, R, M = counts.shape
    pooled_A = counts.sum(axis=1)          # (A, M)
    pooled_R = counts.sum(axis=0)          # (R, M)
    flat = counts.reshape(A * R, M)
    H_X = _H(counts.sum(axis=(0, 1)) / counts.sum())
    acc = {k: [] for k in ("I_AX", "I_RX", "I_ARX", "I_AX_R", "I_RX_A")}

    for _ in range(n_perm):
        sA = _shuffle_table(pooled_A, rng)
        acc["I_AX"].append(H_X - _cond_H(sA[:, None, :], (1,)))
        sR = _shuffle_table(pooled_R, rng)
        acc["I_RX"].append(H_X - _cond_H(sR[None, :, :], (0,)))

        c1 = np.stack([_shuffle_table(counts[:, j], rng) for j in range(R)], axis=1)
        acc["I_AX_R"].append(quantities(c1)["I_AX_R"])

        c2 = np.stack([_shuffle_table(counts[i], rng) for i in range(A)], axis=0)
        acc["I_RX_A"].append(quantities(c2)["I_RX_A"])

        acc["I_ARX"].append(quantities(_shuffle_table(flat, rng).reshape(A, R, M))["I_ARX"])

    return {k: float(np.mean(v)) for k, v in acc.items()}


def decompose(counts, n_perm=400, seed=0, label=""):
    counts = np.asarray(counts, float)
    A, R, M = counts.shape
    q = quantities(counts)
    f = _floors(counts, n_perm, seed)

    chk = abs(q["I_ARX"] - (q["I_AX"] + q["I_RX_A"]))
    chk2 = abs(q["I_ARX"] - (q["I_RX"] + q["I_AX_R"]))

    print(f"{label}  A={A} attitudes, R={R} relationships, M={M} modifiers, n={counts.sum():.0f}")
    print(f"  entropies (bits): H(X)={q['H_X']:.4f}  H(X|A)={q['H_X_A']:.4f}  "
          f"H(X|R)={q['H_X_R']:.4f}  H(X|A,R)={q['H_X_AR']:.4f}")
    print(f"  chain rule residual {max(chk, chk2):.2e}\n")

    print(f"  {'quantity':<12}{'raw':>9}{'floor':>9}{'corrected':>11}")
    for k, nm in (("I_AX", "I(A;X)"), ("I_AX_R", "I(A;X|R)"),
                  ("I_RX", "I(R;X)"), ("I_RX_A", "I(R;X|A)"),
                  ("I_ARX", "I(A,R;X)")):
        print(f"  {nm:<12}{q[k]:>9.4f}{f[k]:>9.4f}{q[k]-f[k]:>11.4f}")

    cA, cAR = q["I_AX"] - f["I_AX"], q["I_AX_R"] - f["I_AX_R"]
    II_c = cA - cAR
    print(f"\n  I(A;R) = {q['I_AR']:.6f} bits   <- factor dependence in the realized design")
    if q["I_AR"] < 1e-6:
        print("     balanced: A and R are independent, so NO confounding is possible.")
        print(f"     I(A;X|R) - I(A;X) = {cAR-cA:+.4f} >= 0 is expected here; it is")
        print("     relationship noise being removed, a power gain, not bias correction.")
    else:
        print(f"     NOT balanced -- confounding is possible. Read II below.")
    thr = 0.05 * max(cA, 1e-9)
    verdict = ("REDUNDANT: relationship explains part of the marginal attitude effect"
               if II_c > thr else
               "SYNERGISTIC: conditioning on relationship reveals more attitude effect"
               if II_c < -thr else
               "additive: the factors barely interact")
    print(f"  II = I(A;X) - I(A;X|R) = {II_c:+.4f} bits (floor-corrected)")
    print(f"     -> {verdict}")
    matters = "conditioning matters" if abs(II_c) > 0.1 * max(cA, 1e-9) else "either is fine"
    print(f"  report I(A;X|R) = {cAR:.4f} bits as the attitude effect ({matters})\n")

    return dict(**q, floors=f, I_AX_corrected=cA, I_AX_R_corrected=cAR, II_corrected=II_c)
df = en_df
MODIFIERS = EN_MODIFIERS
axis = "attitude"
mix = {m: i for i, m in enumerate(MODIFIERS)}
participants = sorted(df["doc_id"].astype(str).unique())
levels = sorted(df[axis].astype(str).unique())
pix = {p: i for i, p in enumerate(participants)}
aix = {a: i for i, a in enumerate(levels)}
counts = np.zeros((len(participants), len(levels), len(MODIFIERS)))   # dimensions is [participant, attitude, modifier]
np.add.at(counts, (df["doc_id"].astype(str).map(pix).to_numpy(),
                    df[axis].astype(str).map(aix).to_numpy(),
                    df["modifier"].map(mix).to_numpy()), 1)
decompose(counts) 
df = jp_df
MODIFIERS = JP_MODIFIERS
axis = "attitude"
mix = {m: i for i, m in enumerate(MODIFIERS)}
participants = sorted(df["doc_id"].astype(str).unique())
levels = sorted(df[axis].astype(str).unique())
pix = {p: i for i, p in enumerate(participants)}
aix = {a: i for i, a in enumerate(levels)}
counts = np.zeros((len(participants), len(levels), len(MODIFIERS)))   # dimensions is [participant, attitude, modifier]
np.add.at(counts, (df["doc_id"].astype(str).map(pix).to_numpy(),
                    df[axis].astype(str).map(aix).to_numpy(),
                    df["modifier"].map(mix).to_numpy()), 1)
decompose(counts) 

  A=10 attitudes, R=7 relationships, M=31 modifiers, n=530
  entropies (bits): H(X)=3.8163  H(X|A)=2.8645  H(X|R)=3.4324  H(X|A,R)=2.1981
  chain rule residual 0.00e+00

  quantity          raw    floor  corrected
  I(A;X)         0.9519   0.3512     0.6007
  I(A;X|R)       1.2342   0.9949     0.2394
  I(R;X)         0.3840   0.2295     0.1544
  I(R;X|A)       0.6663   0.6373     0.0290
  I(A,R;X)       1.6182   1.3322     0.2860

  I(A;R) = 0.039172 bits   <- factor dependence in the realized design
     NOT balanced -- confounding is possible. Read II below.
  II = I(A;X) - I(A;X|R) = +0.3613 bits (floor-corrected)
     -> REDUNDANT: relationship explains part of the marginal attitude effect
  report I(A;X|R) = 0.2394 bits as the attitude effect (conditioning matters)

  A=10 attitudes, R=6 relationships, M=31 modifiers, n=461
  entropies (bits): H(X)=4.3665  H(X|A)=3.4722  H(X|R)=3.5190  H(X|A,R)=2.2144
  chain rule residual 0.00e+00

  quantity          raw    floor  corrected
  I(

{'H_X': 4.366510081179899,
 'H_X_A': 3.472224805447037,
 'H_X_R': 3.5189710919672232,
 'H_X_AR': 2.2144457674137863,
 'I_AX': 0.8942852757328623,
 'I_RX': 0.8475389892126759,
 'I_ARX': 2.152064313766113,
 'I_AX_R': 1.304525324553437,
 'I_RX_A': 1.2577790380332505,
 'II': -0.41024004882057463,
 'I_AR': 0.05907293555915594,
 'floors': {'I_AX': 0.42294987747261414,
  'I_RX': 0.2549325541115106,
  'I_ARX': 1.6551667975860533,
  'I_AX_R': 1.0698294202756111,
  'I_RX_A': 0.9285377038137852},
 'I_AX_corrected': 0.4713353982602482,
 'I_AX_R_corrected': 0.23469590427782583,
 'II_corrected': 0.23663949398242234}